###**Programación Concurrente**
####Actividad Práctica (Opcional) - Comunicación y Sincronización

---

##**Ejercicio 3 - Baño compartido**

In [ ]:
%%writefile bano_compartido.py
import random
import sys
import threading
import time

# Constantes del baño
CAPACITY = 3           # empleados simultaneos dentro del baño
MAX_CONSECUTIVE_USES = 3    # ingresos seguidos del mismo genero (anti-inanicion)

# Tiempos de simulación en segundos
MIN_WORKING_TIME = 0.1
MAX_WORKING_TIME = 0.8
MIN_BATHROOM_TIME = 0.3
MAX_BATHROOM_TIME = 0.9

# Géneros
MEN = "Hombre"
WOMEN = "Mujer"

# Índices de argumentos para el main
ARG_MEN_COUNT = 1
ARG_WOMEN_COUNT = 2

# Valores por defecto para el main
DEFAULT_MEN_COUNT = 6
DEFAULT_WOMEN_COUNT = 6

def opposite_gender(gender):
    return WOMEN if gender == MEN else MEN


class Bathroom:

    def __init__(self):
        self._cond = threading.Condition()
        self._inside = 0               # empleados dentro ahora
        self._current_gender = None      # None = baño vacio
        self._last_gender = None
        self._consecutive_uses = 0
        self._waiting = {MEN: 0, WOMEN: 0}
        # Estadisticas
        self._max_simultaneous = 0
        self._uses = 0

    def enter(self, name, gender):
        with self._cond:
            self._waiting[gender] += 1
            self._log(name, gender, f"espera afuera (adentro: {self._inside})")

            # wait_for vuelve a evaluar el predicado en cada notificacion
            self._cond.wait_for(lambda: self._can_enter(gender))

            self._waiting[gender] -= 1

            if gender != self._last_gender:   # cambia el turno de genero
                self._consecutive_uses = 0
                self._last_gender = gender
            self._consecutive_uses += 1

            self._inside += 1
            self._uses += 1
            self._current_gender = gender
            self._max_simultaneous = max(self._max_simultaneous, self._inside)

            self._log(name, gender, f"ENTRA al baño (adentro: {self._inside})")

    def _can_enter(self, gender):
        """Condicion de seguridad. Se evalua siempre con el lock tomado."""
        if self._inside >= CAPACITY:
            return False
        if self._inside > 0 and self._current_gender != gender:
            return False
        consecutive = self._consecutive_uses if gender == self._last_gender else 0
        if consecutive >= MAX_CONSECUTIVE_USES and self._waiting[opposite_gender(gender)] > 0:
            return False    # cedo el turno al otro grupo
        return True


    def exit(self, name, gender):
        with self._cond:
            self._inside -= 1
            if self._inside == 0:
                self._current_gender = None   # baño libre: puede cambiar de genero
            self._log(name, gender, f"sale del baño  (adentro: {self._inside})")
            self._cond.notify_all()

    def _log(self, name, gender, message):
        print(f"[{name:>10}] {gender:<7} {message}")

    def statistics(self):
        print()
        print("== Resumen ==")
        print(f"Usos totales del baño          : {self._uses}")
        print(f"Maximo de empleados simultaneos: {self._max_simultaneous} (limite {CAPACITY})")


def employee(bathroom, name, gender):
    """Hilo empleado: hace su trabajo y cada tanto usa el baño."""
    for _ in range(1): # El bucle for tiene un rango fijo de 1 iteración.
        time.sleep(random.uniform(MIN_WORKING_TIME, MAX_WORKING_TIME))        # trabajando
        bathroom.enter(name, gender)
        try:
            time.sleep(random.uniform(MIN_BATHROOM_TIME, MAX_BATHROOM_TIME))    # usando el baño
        finally:
            bathroom.exit(name, gender)


def main():
    men_count = int(sys.argv[ARG_MEN_COUNT]) if len(sys.argv) > ARG_MEN_COUNT - 1 else DEFAULT_MEN_COUNT
    women_count = int(sys.argv[ARG_WOMEN_COUNT]) if len(sys.argv) > ARG_WOMEN_COUNT - 1 else DEFAULT_WOMEN_COUNT

    print(f"Hombres: {men_count} | Mujeres: {women_count} | Capacidad del baño: {CAPACITY}\n")

    bathroom = Bathroom()
    threads = []

    for i in range(1, men_count + 1):
        threads.append(threading.Thread(target=employee, args=(bathroom, f"H-{i}", MEN)))
    for i in range(1, women_count + 1):
        threads.append(threading.Thread(target=employee, args=(bathroom, f"M-{i}", WOMEN)))

    for t in threads:
        t.start()
    for t in threads:
        t.join()

    bathroom.statistics()
    print("Todos los empleados usaron el baño sin mezclarse.")


if __name__ == "__main__":
    main()


Writing bano_compartido.py


In [ ]:
!python3 bano_compartido.py 6 6

Hombres: 6 | Mujeres: 6 | Capacidad del baño: 3

[       M-2] Mujer   espera afuera (adentro: 0)
[       M-2] Mujer   ENTRA al baño (adentro: 1)
[       H-4] Hombre  espera afuera (adentro: 1)
[       M-6] Mujer   espera afuera (adentro: 1)
[       M-6] Mujer   ENTRA al baño (adentro: 2)
[       M-3] Mujer   espera afuera (adentro: 2)
[       M-3] Mujer   ENTRA al baño (adentro: 3)
[       H-3] Hombre  espera afuera (adentro: 3)
[       H-1] Hombre  espera afuera (adentro: 3)
[       H-2] Hombre  espera afuera (adentro: 3)
[       M-1] Mujer   espera afuera (adentro: 3)
[       H-5] Hombre  espera afuera (adentro: 3)
[       H-6] Hombre  espera afuera (adentro: 3)
[       M-6] Mujer   sale del baño  (adentro: 2)
[       M-2] Mujer   sale del baño  (adentro: 1)
[       M-5] Mujer   espera afuera (adentro: 1)
[       M-4] Mujer   espera afuera (adentro: 1)
[       M-3] Mujer   sale del baño  (adentro: 0)
[       H-4] Hombre  ENTRA al baño (adentro: 1)
[       H-1] Hombre  ENTRA al baño (

### Conclusiones
Para el problema del baño compartido abordamos una variante de sincronización condicional y aforo limitado. La solución requirió delimitar el baño como una región crítica gobernada por dos restricciones concurrentes: la exclusión mutua entre géneros distintos y un límite de capacidad estricto de tres accesos simultáneos.
Mientras que habilitar ingresos sucesivos del mismo género maximiza el caudal de ocupación y reduce la sobrecarga por conmutación, un flujo ininterrumpido puede inducir starvation en el género opuesto. Por lo que, la coordinación del cambio de turno al desocuparse el baño demostró ser el punto central para asegurar el progreso y la espera limitada sin incurrir a deadlock.

Al comparar los ejercicios 2 y 3, el uso de lenguajes compilados (Java) frente a interpretados (Python) evidencia cómo cada entorno gestiona el no determinismo sobre los recursos compartidos. En plataformas compiladas, las primitivas de sincronización operan sin espera activa y suspenden los hilos directamente en colas del sistema operativo, garantizando la atomicidad y la mutua exclusión en la región crítica, aunque a costa de un código más extenso y estructurado. En contraste, Python abstrae estos mecanismos mediante una sintaxis más compacta, delegando la coordinación en el intérprete.